# Figure 2 — what carries cross-species homology

Connectivity together with an anchor-warped spatial scaffold appears to set region-level
correspondence, while the curated anchors and packs add precision at the parcel level. This
notebook rebuilds that result.

Figure 2 mixes two kinds of evidence, and they are not equally reproducible here:

| | what | cost | handled how |
|---|---|---|---|
| scoring | the frozen coupling against the Beauchamp benchmark | minutes | recomputed below |
| re-fits | ablation ladder, leave-one-region-out, held-out arm | hours | producing script, under `RUN_REFITS` |

An ablation study re-fits the model once per condition, so it cannot be derived inside a notebook
cell. What the notebook does instead is recompute every scoring result from the coupling, and
decline to use a re-fit log whose coupling provenance does not match.

Before running: `python scripts/fetch_data.py`. §1 needs nothing else.

In [ ]:
import importlib.util, json, subprocess, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIGS = ROOT.parent / 'manuscript' / 'figures'
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(FIGS))

from homer.data import load_cached, load_pi, pi_provenance

pi = load_pi()
M, _ = load_cached('mouse', cache_dir=str(ROOT / 'outputs/anndata'))
H, _ = load_cached('human', cache_dir=str(ROOT / 'outputs/anndata'))
PROV = pi_provenance()
print(f"coupling {pi.shape[0]:,} x {pi.shape[1]:,}   {PROV['pi_file']}   sha {PROV['pi_sha256'][:16]}...")

LOGS = ROOT / 'outputs' / 'logs'

# Values as printed in the manuscript; recomputed below rather than read from a results file.
PUBLISHED = {
    'Beauchamp AUROC (parcel-weighted)': (0.90, 0.01),
    'Beauchamp top-1 (parcel-weighted)': (0.57, 0.01),
    'mean centroid displacement (mm)':   (7.7,  0.3),
    'ladder AUROC, connectivity only':   (0.69, 0.01),
    'ladder AUROC, + spatial':           (0.97, 0.01),
    'ladder AUROC, + packs (production)':(0.90, 0.01),
    'LORO held-out mean AUROC':          (0.74, 0.01),
}

def check(name, value):
    exp, tol = PUBLISHED[name]
    ok = abs(value - exp) <= tol
    print(f"  [{'ok ' if ok else 'FAIL'}] {name:36s} computed {value:.4f}   manuscript {exp}")
    return ok


def verified_log(fname):
    """Read a re-fit log ONLY after confirming which coupling produced it.

    A log named '..._canonical.json' is not evidence that it was built on the canonical coupling;
    that assumption is exactly what produced the July 2026 false all-clear. Where the log carries
    a sha it must match; where it carries none the notebook says so rather than staying quiet.
    """
    d = json.loads((LOGS / fname).read_text())
    sha = d.get('pi_sha256')
    if sha is None:
        print(f"  {fname}: NO coupling provenance recorded -- cannot verify; re-fit to be certain")
    elif sha != PROV['pi_sha256']:
        raise RuntimeError(f"{fname} was built on a DIFFERENT coupling ({sha[:16]}...)")
    else:
        print(f"  {fname}: provenance verified against canonical sha")
    return d

## 1. Scoring against an external benchmark (Fig. 2b, 2d)

The Beauchamp set is 19 mouse→human region pairs curated from the comparative literature, which
the model never saw. Scoring is cheap because the coupling is frozen: we ask where each mouse
region's mass lands. The cell below recomputes the production battery from π.

Aggregates are weighted by parcel count. Regions differ roughly fifty-fold in size, so an
unweighted mean would let a 5-parcel region count as much as a 250-parcel one. Both are printed,
since the difference (0.90 against 0.93) is easy to trip over.

In [ ]:
# The scorer used by the panel scripts, imported rather than reimplemented.
_bb = ROOT / 'experiments' / 'section2_supervision' / 'beauchamp_battery.py'
_spec = importlib.util.spec_from_file_location('bb', _bb)
BB = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(BB)

pairs, reg_cents, reg_masks, h_xyz, brain_c, _ = BB.build(M, H)
res = BB.score_all(pi.astype(np.float64), pairs, reg_cents, reg_masks, h_xyz, brain_c)
per = res['per_region']

keys = list(per)
w = np.array([per[k]['n_mouse'] for k in keys], float)
auroc = np.array([per[k]['auroc'] for k in keys])
top1 = np.array([per[k]['top1'] for k in keys])
disp = np.array([per[k]['centroid_disp_mm'] for k in keys])
chance = np.array([per[k]['expected_disp_mm'] for k in keys])

print(f"{len(keys)} benchmark pairs scored on the canonical coupling\n")
print(f"AUROC  parcel-weighted {np.average(auroc, weights=w):.3f}   unweighted {auroc.mean():.3f}")
print(f"top-1  parcel-weighted {np.average(top1, weights=w):.3f}   unweighted {top1.mean():.3f}")
print(f"displacement {disp.mean():.1f} mm observed vs {chance.mean():.1f} mm chance\n")
check('Beauchamp AUROC (parcel-weighted)', float(np.average(auroc, weights=w)))
check('Beauchamp top-1 (parcel-weighted)', float(np.average(top1, weights=w)))
check('mean centroid displacement (mm)', float(disp.mean()))

sig = sum(1 for k in keys if per[k].get('perm_q_mass', 1) < 0.05)
print(f"\nmass enrichment significant (parcel-set permutation, FDR q<0.05): {sig}/{len(keys)}")

In [ ]:
# The chance line on Fig. 2d, derived rather than asserted. A region carrying no spatial
# information would place its centroid at the human parcel-cloud centroid, so its chance
# displacement is the distance from the true homologue centroid to that point.
fig, ax = plt.subplots(figsize=(6.0, 6.4))
order = np.argsort(auroc)
names = [k.split(' -> ')[0].replace('Primary ', '') for k in keys]
y = np.arange(len(order))
ax.barh(y, disp[order], color=['#1b6f9c' if per[keys[i]].get('spin_q_disp', 1) < 0.05 else '#c9c9c9'
                               for i in order], height=0.66)
ax.axvline(chance.mean(), ls='--', lw=1, color='#888')
ax.text(chance.mean(), len(y) - 0.2, f'chance {chance.mean():.0f} mm', fontsize=7.5,
        color='#888', ha='center')
ax.set_yticks(y); ax.set_yticklabels([names[i] for i in order], fontsize=8)
ax.set_xlabel('centroid displacement (mm)')
ax.set_title('Predictions land within region-level distance\nof the true homologue',
             fontweight='bold', loc='left', fontsize=10.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.show()

## 2. What does each cost term buy? (Fig. 2a)

The decomposition extends connectivity alone (Gromov–Wasserstein on FC and SC) with the
anchor-warped spatial scaffold, then the curated anchors, then the region packs, re-fitting a
coupling at each stage.

Because it is a re-fit, it is not derived here. Set `RUN_REFITS = True` to rebuild it, or leave it
off to verify and read the existing log.

One caveat worth keeping in view: the spatial scaffold is itself fitted to the Garin landmark
pairs, so it is not supervision-free. The ladder separates kinds of supervision rather than
supervision from none.

In [ ]:
RUN_REFITS = False        # True re-fits every ablation stage from scratch (hours)

LADDER = ROOT / 'experiments' / 'section5_coverage_rigor' / '35_ablation_ladder_canonical.py'

if RUN_REFITS:
    print(f're-fitting the ablation ladder via {LADDER.name} ...')
    r = subprocess.run([sys.executable, str(LADDER)], cwd=str(ROOT), capture_output=True, text=True)
    print((r.stdout or r.stderr)[-800:])

ladder = verified_log('ablation_ladder_battery_canonical.json')
stages = ['connectivity', '+spatial', '+anchors', '+packs']
print()
for s in stages:
    v = ladder[s]
    print(f"  {s:14s} AUROC {v['auroc']:.3f}   top-1 {v['top1']:.3f}   "
          f"mass {v['mass_in_region']:.3f}   disp {v['centroid_disp_mm']:.1f} mm")
print()
check('ladder AUROC, connectivity only', ladder['connectivity']['auroc'])
check('ladder AUROC, + spatial', ladder['+spatial']['auroc'])
check('ladder AUROC, + packs (production)', ladder['+packs']['auroc'])

print("\nRegion-level recovery jumps once space is added and does not improve with curation;")
print("parcel-exact recovery moves only when the anchors and packs arrive. Two different things.")

## 3. What happens if the curation is withheld? (Fig. 2c, 2e)

Two held-out tests, both re-fits:

- leave-one-region-out across all 41 combined supervision units (15 Garin classes, 26 packs):
  remove one, re-fit, and score the held-out unit from connectivity and space alone;
- a memorisation control: for each of the 19 Beauchamp regions, remove the curation overlapping
  it, re-fit, and re-score.

If agreement with the benchmark were memorised curation, both should collapse. Region-level
recovery largely holds. Parcel-exact recovery does collapse, which bounds the claim.

In [ ]:
LORO_SCRIPTS = {
    'anchor_recovery_loo_combined_canonical.json': ROOT / 'experiments' / 'section2_supervision' / 'anchor_recovery_loo_canonical.py',
    'beauchamp_metric_battery_loro_canonical.json': ROOT / 'experiments' / 'section2_supervision' / 'beauchamp_battery_canonical.py',
}

if RUN_REFITS:
    for log, script in LORO_SCRIPTS.items():
        args = [sys.executable, str(script)] + (['--loro'] if 'beauchamp' in script.name else [])
        print(f're-fitting {log} via {script.name} ...')
        r = subprocess.run(args, cwd=str(ROOT), capture_output=True, text=True)
        print((r.stdout or r.stderr)[-400:])

loo = verified_log('anchor_recovery_loo_combined_canonical.json')
# The log is keyed by unit name ('Garin:...' / 'Pack:...'), each carrying the held-out AUROC
# scored from connectivity and space alone. Same parse as make_fig2c_loro.py.
unit_names = list(loo)
ho = np.array([loo[k]['auroc'] for k in unit_names], float)
is_garin = np.array([k.startswith('Garin') for k in unit_names])
print(f"\n{len(ho)} supervision units held out "
      f"({int(is_garin.sum())} Garin classes, {int((~is_garin).sum())} region packs)")
print(f"  held-out mean AUROC {ho.mean():.3f}   below chance {(ho < 0.5).sum()}/{len(ho)}")
print(f"  Garin {ho[is_garin].mean():.3f}   packs {ho[~is_garin].mean():.3f}")
check('LORO held-out mean AUROC', float(ho.mean()))

In [ ]:
# The memorisation control: production vs held-out AUROC on the 19 benchmark regions.
lo = verified_log('beauchamp_metric_battery_loro_canonical.json')
lo = {k: v for k, v in lo.items() if isinstance(v, dict) and 'auroc' in v}
common = [k for k in keys if k in lo]
a_full = np.array([per[k]['auroc'] for k in common])
a_ho = np.array([lo[k]['auroc'] for k in common])
wv = np.array([per[k]['n_mouse'] for k in common], float)

print(f"{len(common)} regions scored both ways")
print(f"  parcel-weighted AUROC {np.average(a_full, weights=wv):.3f} production "
      f"-> {np.average(a_ho, weights=wv):.3f} held-out")
print(f"  unweighted            {a_full.mean():.3f} -> {a_ho.mean():.3f}")
print(f"  below chance held-out {(a_ho < 0.5).sum()}/{len(a_ho)}")
print("\nRecovery is largely retained, so agreement is not memorised curation.")

## 4. Build the figure panels

Each panel has one producing script in `manuscript/figures/fig2/`. They read the logs verified
above and write into that folder.

In [ ]:
RUN_PANELS = True

PANELS = [
    ('a', 'make_fig2a_ladder.py'),
    ('b', 'make_fig2b_heatgrid.py'),
    ('c', 'make_fig2c_loro.py'),
    ('d, e', 'make_fig2de_beauchamp.py'),
]

if RUN_PANELS:
    for panel, script in PANELS:
        r = subprocess.run([sys.executable, str(FIGS / 'fig2' / script)], cwd=str(ROOT),
                           capture_output=True, text=True)
        status = 'ok' if r.returncode == 0 else f'FAILED ({r.returncode})'
        print(f"panel {panel:6s} {script:28s} {status}")
        if r.returncode != 0:
            print((r.stderr or '')[-400:])
else:
    print('skipped; set RUN_PANELS = True')

## 5. Summary

§1 was recomputed from the coupling. §2 and §3 are re-fits: their logs were checked for coupling
provenance before being read, and can be rebuilt with `RUN_REFITS = True`.

In [ ]:
print(f"coupling                      {PROV['pi_file']}")
print(f"Beauchamp AUROC               {np.average(auroc, weights=w):.2f} parcel-weighted (recomputed)")
print(f"Beauchamp top-1               {np.average(top1, weights=w):.2f} parcel-weighted (recomputed)")
print(f"displacement                  {disp.mean():.1f} mm vs {chance.mean():.1f} mm chance (recomputed)")
print(f"ladder AUROC                  {ladder['connectivity']['auroc']:.2f} -> "
      f"{ladder['+spatial']['auroc']:.2f} -> {ladder['+packs']['auroc']:.2f} (re-fit)")
print(f"LORO held-out mean            {ho.mean():.2f} over {len(ho)} units (re-fit)")
print()
ok = all([check('Beauchamp AUROC (parcel-weighted)', float(np.average(auroc, weights=w))),
          check('Beauchamp top-1 (parcel-weighted)', float(np.average(top1, weights=w))),
          check('mean centroid displacement (mm)', float(disp.mean())),
          check('ladder AUROC, connectivity only', ladder['connectivity']['auroc']),
          check('ladder AUROC, + spatial', ladder['+spatial']['auroc']),
          check('ladder AUROC, + packs (production)', ladder['+packs']['auroc']),
          check('LORO held-out mean AUROC', float(ho.mean()))])
print('\nALL CHECKS PASS' if ok else '\nSOME CHECKS FAILED -- text and code have diverged')